# 07 — Construction d'Équipe Optimale pour chaque génération

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import pickle
import requests
from math import pi

plt.rcParams.update({
    'figure.dpi': 130,
    'figure.facecolor': 'white',
    'axes.facecolor': '#F5F6FA',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.35,
    'grid.linestyle': '--',
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

STATS = ['hp', 'attack', 'defense', 'special-attack', 'special-defense', 'speed']
TYPE_COLORS = {
    'normal': '#A8A878', 'fire': '#F08030', 'water': '#6890F0', 'electric': '#F8D030',
    'grass': '#78C850', 'ice': '#98D8D8', 'fighting': '#C03028', 'poison': '#A040A0',
    'ground': '#E0C068', 'flying': '#A890F0', 'psychic': '#F85888', 'bug': '#A8B820',
    'rock': '#B8A038', 'ghost': '#705898', 'dragon': '#7038F8', 'dark': '#705848',
    'steel': '#B8B8D0', 'fairy': '#EE99AC', 'stellar': '#40B5A5',
}
TEAM_COLORS = ['#E74C3C','#3498DB','#2ECC71','#F39C12','#9B59B6','#1ABC9C']

with open('../data/pokemon_full.pkl', 'rb') as f:
    data = pickle.load(f)
df, matrix, ALL_TYPES = data['df'], data['matrix'], data['ALL_TYPES']
print(f'Données chargées : {df.shape[0]} Pokémon')

gen_map = {
    'generation-i': 1, 'generation-ii': 2, 'generation-iii': 3,
    'generation-iv': 4, 'generation-v': 5, 'generation-vi': 6,
    'generation-vii': 7, 'generation-viii': 8, 'generation-ix': 9
}
df['generation'] = df['generation'].map(gen_map)

for gen, count in df['generation'].value_counts().sort_index().items():
    print(f"  Génération {int(gen)} : {count} Pokémon")

Données chargées : 1025 Pokémon
  Génération 1 : 151 Pokémon
  Génération 2 : 100 Pokémon
  Génération 3 : 135 Pokémon
  Génération 4 : 107 Pokémon
  Génération 5 : 156 Pokémon
  Génération 6 : 72 Pokémon
  Génération 7 : 88 Pokémon
  Génération 8 : 96 Pokémon
  Génération 9 : 120 Pokémon


# Génération 1

## 1. Filtrage des pokémon

In [11]:
def compute_individual_scores(df, target_gen):
    gen_df = df[df['generation'] == target_gen].copy()
    gen_df['indiv_score'] = (
        gen_df['bst'].fillna(0)
        + 12 * gen_df['n_offense'].fillna(0)
        + 4  * gen_df['n_resistances'].fillna(0)
        - 6  * gen_df['n_weaknesses'].fillna(0)
        + 8  * gen_df['defense_score'].fillna(0)
    )
    return gen_df

## 2. Extraction du pool de candidats

In [12]:
def get_candidates(gen_df, pool_size=24):
    candidates = gen_df.nlargest(min(pool_size, len(gen_df)), 'indiv_score')
    return candidates.reset_index(drop=True)

## 3. Préparation des données pour le team building

In [13]:
def precompute_arrays(candidates, ALL_TYPES):
    bst_arr    = candidates['bst'].to_numpy(dtype=float)
    type1_arr  = candidates['type1'].fillna('').to_numpy()
    type2_arr  = candidates['type2'].fillna('').to_numpy()

    offense_sets = []
    for cov in candidates['offense_coverage']:
        if isinstance(cov, (set, list, tuple, np.ndarray)):
            offense_sets.append(set(cov))
        else:
            offense_sets.append(set())

    defense_arr = np.ones((len(candidates), len(ALL_TYPES)), dtype=float)
    for i, prof in enumerate(candidates['defense_profile']):
        if isinstance(prof, dict):
            defense_arr[i] = [float(prof.get(t, 1.0)) for t in ALL_TYPES]

    return bst_arr, type1_arr, type2_arr, offense_sets, defense_arr

## 4. Score team

In [14]:
def score_team(idxs, bst_arr, type1_arr, type2_arr, offense_sets, defense_arr):
    idxs   = np.array(idxs, dtype=int)
    power  = bst_arr[idxs].mean()

    cov    = set().union(*[offense_sets[i] for i in idxs])
    offense = len(cov) * 16

    team_types = {t for i in idxs for t in (type1_arr[i], type2_arr[i]) if t}
    diversity  = len(team_types) * 5

    m           = defense_arr[idxs]
    weak        = (m > 1).sum(axis=0)
    resist      = ((m < 1) & (m > 0)).sum(axis=0)
    immune      = (m == 0).sum(axis=0)
    weak_stacks = np.clip(weak - 2, 0, None).sum()
    defense     = resist.sum() * 1.3 + immune.sum() * 4 - weak_stacks * 10

    unique_primary = len({type1_arr[i] for i in idxs if type1_arr[i]})
    dup_penalty    = (len(idxs) - unique_primary) * 6

    return power + offense + diversity + defense - dup_penalty


## 5. Construction de la meilleure équipe

In [15]:
def build_best_team_df(candidates, best_idxs):
    team = candidates.iloc[list(best_idxs)].copy()
    team['pokemon'] = team['name'].str.replace('-', ' ').str.title()
    return team.sort_values('bst', ascending=False).reset_index(drop=True)

## 6. Visualisation de l'équipe optimale

In [18]:
def plot_radar(best_team_df, STATS, TEAM_COLORS, title):
    stats   = best_team_df.set_index('pokemon')[STATS]
    angles  = np.linspace(0, 2 * pi, len(STATS), endpoint=False).tolist() + [0]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'polar': True})
    for i, (name, row) in enumerate(stats.iterrows()):
        ax.plot(angles, row.tolist() + [row.iloc[0]],
                color=TEAM_COLORS[i % len(TEAM_COLORS)], linewidth=2, label=name)

    mean_vals = stats.mean().tolist() + [stats.mean().iloc[0]]
    ax.plot(angles, mean_vals, color='#2C3E50', linewidth=3, linestyle='--', label='Moyenne équipe')
    ax.fill(angles, mean_vals, color='#2C3E50', alpha=0.12)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([s.replace('-', ' ').title() for s in STATS])
    ax.set_ylim(0, max(160, int(stats.to_numpy().max() + 20)))
    ax.set_title(title, pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.38, 1.12), frameon=False)
    plt.tight_layout()
    plt.show()

In [ ]:
# Affiche le graphique radar de l'équipe optimale
plot_radar(
    best_team_df,
    STATS,
    TEAM_COLORS,
    title="Équipe optimale — Génération 1"
)